# Interactive hackingtool workspace

This notebook is ready for **Run All** in the repository's GitHub Codespace. Commands run as `root` inside the isolated container, not on the Codespaces host.

The notebook prepares a non-blocking launcher and a safe argument-based command helper. Run security tools only against systems you own or are explicitly authorized to test.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from typing import Sequence


def find_repo_root(start: Path | None = None) -> Path:
    """Find the repository without depending on the notebook launch directory."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "hackingtool.py").is_file():
            return candidate
    raise FileNotFoundError("Could not find hackingtool.py from the current directory")


REPO_ROOT = find_repo_root()
IS_ROOT = (os.geteuid() == 0) if hasattr(os, "geteuid") else False
IS_ISOLATED_CONTAINER = Path("/.dockerenv").exists() or bool(
    os.environ.get("CODESPACES") or os.environ.get("REMOTE_CONTAINERS")
)


def run_command(
    arguments: Sequence[str | os.PathLike[str]],
    *,
    cwd: Path = REPO_ROOT,
    check: bool = True,
) -> subprocess.CompletedProcess[str]:
    """Run an explicit argument list without shell interpolation."""
    if not arguments:
        raise ValueError("At least one command argument is required")
    command = [os.fspath(argument) for argument in arguments]
    print("Running:", subprocess.list2cmdline(command))
    return subprocess.run(
        command,
        cwd=cwd,
        check=check,
        text=True,
    )


def launch_hackingtool() -> subprocess.CompletedProcess[str]:
    """Launch the interactive CLI on demand; this is not called by Run All."""
    return run_command([sys.executable, REPO_ROOT / "hackingtool.py"], check=False)

In [ ]:
# Run-All readiness check.
required_files = [
    REPO_ROOT / "hackingtool.py",
    REPO_ROOT / "constants.py",
    REPO_ROOT / "requirements.txt",
]
missing_files = [path.name for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f"Missing required project files: {missing_files}")

write_probe = REPO_ROOT / ".interactive-notebook-write-check"
try:
    write_probe.write_text("ok", encoding="utf-8")
finally:
    write_probe.unlink(missing_ok=True)

print(f"Repository:          {REPO_ROOT}")
print(f"Python:              {sys.version.split()[0]}")
print(f"Root in container:   {IS_ROOT}")
print(f"Container detected:  {IS_ISOLATED_CONTAINER}")
print("\nRun All is complete. Call launch_hackingtool() when you want the interactive CLI.")